In [11]:
from dataclasses import dataclass

@dataclass
class CPU:
    brand: str
    socket: str
    memory_type: str

@dataclass
class Motherboard:
    socket: str
    supported_memory: str

@dataclass
class RAM:
    memory_type: str
    capacity: str

@dataclass
class PCConfig:
    cpu: str = None
    motherboard: str = None
    ram: str = None

In [7]:
def rule_cpu_socket(pc: "PCConfig"):
    if pc.cpu:
        return {"socket": pc.cpu.socket}
    return {}

def rule_cpu_memory_type(pc: "PCConfig"):
    if pc.cpu:
        return {"supported_memory": pc.cpu.memory_type}
    return {}

def rule_motherboard_memory_type(pc: "PCConfig"):
    if pc.motherboard:
        return {"memory_type": {"$in": pc.motherboard.supported_memory}}
    return {}

In [8]:
FILTER_RULES = {
    "motherboard": [
        rule_cpu_socket,
        rule_cpu_memory_type,
    ],
    "ram": [
        rule_cpu_memory_type,
        rule_motherboard_memory_type,
    ],
    # аналогічно gpu, psu і т.д.
}

In [15]:
def get_filters(component_type: str, pc: PCConfig, compatibility: bool = True) -> dict:
    rules = FILTER_RULES.get(component_type, [])
    filters = {}

    if not compatibility:
        return {}

    for rule in rules:
        filters.update(rule(pc))

    return filters

In [18]:
pc = PCConfig(
    cpu=CPU("AMD", "AM4", "DDR4")
)

print(get_filters("motherboard", pc))

print(get_filters("ram", pc))

print(get_filters("ram", pc, compatibility=False))

{'socket': 'AM4', 'supported_memory': 'DDR4'}
{'supported_memory': 'DDR4'}
{}
